# Project LAWA — Fecal coliform soft sensor models

**Goal:** estimate fecal coliform (MPN/100 mL) at the 15 LLDA Laguna Lake stations from field-measurable readings, and compare Multiple Linear Regression (MLR), Random Forest and XGBoost (Chapter 3, Specific Objective 2). As in the manuscript, the model works on **log10** values and the estimates are converted back to MPN/100 mL.

**How to run**
1. Put `fc_class_dataset.csv` in Google Drive, in a folder named **Project LAWA** (My Drive › Project LAWA).
2. Runtime › Run all. Allow access to Google Drive when asked. A full run takes about 10–15 minutes (grid search).

**Contents**
1. Data overview · 2. Preprocessing (done in the workbook) · 3. Model inputs · 4. Train/test split · 5. Models and baselines · 6. Best model in detail · 7. DENR class check · 8. Year-by-year (rolling) test · 9. Random split comparison · 10. Benchmarks · 11. Final model for the dashboard

In [ ]:
# --- Libraries ---
import warnings, json, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn, xgboost
from scipy.stats import ttest_rel
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SEED = 42                 # fixed random seed so results can be reproduced
TEST_START = "2024-01-01" # train on 2016-2023, test on 2024-2025
MAX_LAG_MONTHS = 3        # "last month's value" = previous record of the station, at most 3 months earlier
READINGS = ["DO", "pH", "Ammonia", "Nitrate", "Phosphate", "Temperature"]

print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__, "| pandas", pd.__version__)

In [ ]:
# --- Google Drive: where the CSV files are ---
# Put the CSV file(s) in a Drive folder named "Project LAWA" (My Drive > Project LAWA).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Project LAWA"
except ImportError:                      # running outside Colab (e.g. on a laptop)
    DATA_DIR = os.environ.get("LAWA_DATA_DIR", ".")

OUT_DIR = os.path.join(DATA_DIR, "model_outputs")   # trained models are saved here
os.makedirs(OUT_DIR, exist_ok=True)
print("Reading data from:", DATA_DIR)
print("Files there:", sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv")))

## 1. Data overview
One row per station per month, February 2016 to December 2025 (sheet *Preprocessing v4 (flagged)* of the LLDA workbook). Sources: LLDA compiled workbook (2016 to March 2023), LLDA quarterly report PDFs (April 2023 to December 2025, transcribed), LLDA temperature records.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "fc_class_dataset.csv"))
df = df.rename(columns={"FC_MPN100mL": "FC"})
df["Month"] = pd.to_datetime(df["Month"])
df["Outlier_flag"] = df["Outlier_flag"].fillna("")
df["logFC"] = np.log10(df["FC"])
TARGET_COL, LAST, FILLED = "logFC", "logFC_last", "FC_imputed"

print(f"{len(df)} rows · {df.Station.nunique()} stations · {df.Month.min():%Y-%m} to {df.Month.max():%Y-%m}")
df.head()

In [ ]:
# Summary statistics (fecal coliform is very skewed, hence the log10 scale)
df[READINGS + ["FC", "logFC"]].describe().T.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df.FC, bins=60, color="#777"); axes[0].set_title("Fecal coliform (MPN/100 mL): very skewed")
axes[1].hist(df.logFC, bins=40, color="#1565c0"); axes[1].set_title("log10 fecal coliform: much more even")
plt.tight_layout(); plt.show()

In [ ]:
# DENR fecal coliform classes (WQG 2021-19). Laguna Lake is Class C: above 200 MPN/100 mL = above standard.
FC_LIMITS = [("A", 50), ("B", 100), ("C", 200), ("D", 400)]
CLASSES = ["A", "B", "C", "D", "Failed"]
STANDARD = 200

def fc_class(v):
    for name, upper in FC_LIMITS:
        if v <= upper:
            return name
    return "Failed"

counts = df["FC"].map(fc_class).value_counts().reindex(CLASSES, fill_value=0)
print(pd.DataFrame({"rows": counts, "share": (counts / counts.sum()).round(3)}))
ax = counts.plot.bar(color=["#8ec5ea", "#dddddd", "#f2d64b", "#e06666", "#1a1a1a"], edgecolor="black", rot=0,
                     title="Fecal coliform classes in the dataset")
ax.set_ylabel("station-months"); plt.show()

In [ ]:
# Lake-wide monthly fecal coliform over time (geometric mean of stations)
monthly = 10 ** df.groupby("Month")["logFC"].mean()
ax = monthly.plot(figsize=(11, 3.5), color="black", lw=1, logy=True)
ax.axhline(STANDARD, color="red", ls="--", lw=1, label="Class C limit (200)")
ax.set_ylabel("MPN/100 mL (log scale)"); ax.set_title("Monthly lake-wide fecal coliform — note the drop in 2018"); ax.legend(); plt.show()
print("Median fecal coliform per year:")
print(df.groupby(df.Month.dt.year)["FC"].median())

## 2. Preprocessing (already applied in the workbook)
1. **Rogue data:** physically impossible values treated as errors.
2. **Below detection limit:** "<x" stored as x/2 (same rule as the 2013–2023 workbook).
3. **Missing values:** linear interpolation within the station, gaps of up to 6 months only (`FC_imputed = True`). Filled values are **never used as test targets**.
4. **Outliers:** readings beyond ±3 SD are **flagged, not deleted** (`Outlier_flag`); the most extreme ones were verified against the official LLDA reports.
5. **TSS is not used:** no TSS data for 2016–2023.
6. **log10 transform** of fecal coliform before modelling; estimates are converted back with 10^x.

In [ ]:
print("Filled (interpolated) fecal coliform values:", int(df[FILLED].sum()))
print("Rows with an outlier flag:", int((df.Outlier_flag != "").sum()))
print("\nHighest fecal coliform values (kept):")
df.nlargest(5, "FC")[["Month", "Station"] + READINGS + ["FC"]]

## 3. Model inputs
| Version | Inputs | Predicts |
|---|---|---|
| A | the 6 readings: DO, pH, ammonia, nitrate, inorganic phosphate, temperature | log10 FC |
| B | A + station (one-hot) + month (sine/cosine) | log10 FC |
| C | B + last month's log10 FC at the same station (at most 3 months earlier) | log10 FC |
| D | same inputs as C | the **change** from last month (log10 FC − last month's log10 FC) |

**Version D (differencing):** the model predicts how much fecal coliform moves from last month and adds that to last month's value, so it starts from "repeat last month" and only learns the adjustment.

In [ ]:
# --- Model inputs ---
df = df.sort_values(["Station", "Month"]).reset_index(drop=True)

# last month's lab value of the target at the same station (only if at most 3 months earlier)
prev_month = df.groupby("Station")["Month"].shift(1)
gap = (df.Month.dt.year - prev_month.dt.year) * 12 + (df.Month.dt.month - prev_month.dt.month)
df[LAST] = df.groupby("Station")[TARGET_COL].shift(1).where(gap <= MAX_LAG_MONTHS)

# month as a cycle (December sits next to January)
df["month_sin"] = np.sin(2 * np.pi * df.Month.dt.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df.Month.dt.month / 12)

# station as one-hot columns
stations = pd.get_dummies(df["Station"], prefix="stn", dtype=float)
STN = list(stations.columns)
df = pd.concat([df, stations], axis=1)

before = len(df)
df = df.dropna(subset=[LAST]).sort_values(["Month", "Station"]).reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} (rows without a previous value within {MAX_LAG_MONTHS} months are dropped, "
      "so every version uses exactly the same rows)")

In [ ]:
C_COLS = READINGS + STN + ["month_sin", "month_cos", LAST]
VERSIONS = {  # name: (inputs, predicts the change?)
    "A: 6 readings": (READINGS, False),
    "B: + station + month": (READINGS + STN + ["month_sin", "month_cos"], False),
    "C: + last month's FC": (C_COLS, False),
    "D: change from last month": (C_COLS, True),
}
df["dlogFC"] = df["logFC"] - df[LAST]
for v, (cols, change) in VERSIONS.items():
    print(f"{v}: {len(cols)} input columns, target = {'change in log10 FC' if change else 'log10 FC'}")

## 4. Train/test split
**Time-based:** the models learn from 2016–2023 and are tested on 2024–2025, months they have never seen. Section 9 shows why the random 70:30 split in the manuscript gives over-optimistic scores.

In [ ]:
# --- Time-based split: train 2016-2023, test 2024-2025 ---
train = df[df.Month < TEST_START]
test = df[(df.Month >= TEST_START) & (~df[FILLED])]   # filled (interpolated) values are never used as test targets
print(f"Training rows: {len(train)} ({train.Month.min():%Y-%m} to {train.Month.max():%Y-%m})")
print(f"Test rows:     {len(test)} ({test.Month.min():%Y-%m} to {test.Month.max():%Y-%m}), measured values only")

## 5. Models and baselines
- **Baselines** every model must beat: *guess the training average* and *repeat last month's value*.
- **Tuning:** grid search with time-ordered 5-fold cross-validation on the training years only.
- **Metrics (manuscript):** R² on the log10 scale (Benchmark 2); RMSE and MAE both on log10 and back-transformed to MPN/100 mL.
- **Also reported:** *skill vs repeat last month* (> 0 = better than persistence), share of above-standard months (> 200) caught, and false alarms.

In [ ]:
# --- Models compared (same three as the manuscript) and their tuning grids ---
MODELS = {
    "MLR": (make_pipeline(StandardScaler(), LinearRegression()), {}),
    "Random Forest": (RandomForestRegressor(random_state=SEED, n_jobs=-1),
                      {"n_estimators": [200, 500], "max_depth": [3, 5, 10, None], "min_samples_split": [2, 5, 10]}),
    "XGBoost": (XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
                {"learning_rate": [0.03, 0.1], "max_depth": [2, 3, 4], "n_estimators": [200, 500],
                 "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]}),
}
CV = TimeSeriesSplit(n_splits=5)   # time-ordered: each fold validates on months AFTER its training months


def tune(model, grid, X, y):
    """Grid search on the TRAINING years only. Returns (fitted best model, best settings)."""
    if not grid:
        return clone(model).fit(X, y), {}
    gs = GridSearchCV(model, grid, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X, y)
    return gs.best_estimator_, gs.best_params_


def fold_rmse(model, X, y):
    """RMSE on each time-ordered CV fold of the training years (used for the paired t-test)."""
    out = []
    for tr, va in CV.split(X):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        out.append(np.sqrt(mean_squared_error(y.iloc[va], m.predict(X.iloc[va]))))
    return np.array(out)

In [ ]:
def score(data, pred_log):
    y_log, y, last = data["logFC"].values, data["FC"].values, data[LAST].values
    pred = 10 ** np.asarray(pred_log)
    above_t, above_p = y > STANDARD, pred > STANDARD
    return {"R2_log": r2_score(y_log, pred_log),
            "skill_vs_last_month": 1 - ((y_log - pred_log) ** 2).sum() / ((y_log - last) ** 2).sum(),
            "RMSE_log": np.sqrt(mean_squared_error(y_log, pred_log)), "MAE_log": mean_absolute_error(y_log, pred_log),
            "RMSE_MPN": np.sqrt(mean_squared_error(y, pred)), "MAE_MPN": mean_absolute_error(y, pred),
            "above_std_caught_%": 100 * (above_p & above_t).sum() / max(above_t.sum(), 1),
            "false_alarms_%": 100 * (above_p & ~above_t).sum() / max((~above_t).sum(), 1)}


def fit_predict(model, cols, change, tr, te):
    """Fit on tr, return log10 estimates for te (version D adds the predicted change to last month's value)."""
    target = "dlogFC" if change else "logFC"
    m = clone(model).fit(tr[cols], tr[target])
    p = m.predict(te[cols])
    return m, (te[LAST].values + p if change else p)


rows, preds, fitted, params, folds = [], {}, {}, {}, {}
rows.append({"version": "Baseline", "model": "Guess the average", **score(test, np.full(len(test), train.logFC.mean()))})
rows.append({"version": "Baseline", "model": "Repeat last month", **score(test, test[LAST].values)})

for vname, (cols, change) in VERSIONS.items():
    target = "dlogFC" if change else "logFC"
    for mname, (model, grid) in MODELS.items():
        m, p = tune(model, grid, train[cols], train[target])
        key = (vname, mname)
        fitted[key], params[key] = m, p
        pred = m.predict(test[cols])
        preds[key] = test[LAST].values + pred if change else pred
        folds[key] = fold_rmse(m, train[cols], train[target])   # same size as the level error for version D
        rows.append({"version": vname, "model": mname, **score(test, preds[key])})
        print(f"done: {vname} / {mname} {p}")

results = pd.DataFrame(rows)
results.round(3)

In [ ]:
trained = results[results.version != "Baseline"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, metric, title in zip(axes, ["R2_log", "RMSE_log", "above_std_caught_%"],
                             ["R² on log10 (higher is better)", "RMSE on log10 (lower is better)", "% of above-standard months caught"]):
    trained.pivot(index="version", columns="model", values=metric).loc[list(VERSIONS)].plot.bar(
        ax=ax, rot=0, color=["#9e9e9e", "#2e7d32", "#1565c0"])
    for _, b in results[results.version == "Baseline"].iterrows():
        ax.axhline(b[metric], ls="--", lw=1, color="red" if "last" in b.model else "black", label=f"baseline: {b.model}")
    if metric == "R2_log":
        ax.axhline(0.55, color="orange", lw=1.5, label="Benchmark 2 (0.55)")
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", labelsize=7); ax.legend(fontsize=7)
fig.suptitle("Fecal coliform models on 2024–2025 (never seen in training)"); plt.tight_layout(); plt.show()

## 6. Best model in detail
The best model is the one with the highest test R² on the log10 scale.

In [ ]:
best = trained.sort_values("R2_log", ascending=False).iloc[0]
BEST = (best.version, best.model)
best_cols, best_change = VERSIONS[best.version]
p_best = preds[BEST]
y_log = test.logFC.values
print(f"Best: {best.version} / {best.model}  ->  R² (log) {best.R2_log:.3f}, RMSE (log) {best.RMSE_log:.3f}, "
      f"skill vs last month {best.skill_vs_last_month:+.3f}")
print("Settings:", params[BEST])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_log, p_best, s=12, alpha=0.5)
lim = [min(y_log.min(), p_best.min()) - 0.2, max(y_log.max(), p_best.max()) + 0.2]
axes[0].plot(lim, lim, "k--", lw=1); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
for e in [50, 100, 200, 400]:
    axes[0].axvline(np.log10(e), color="0.85", lw=0.8); axes[0].axhline(np.log10(e), color="0.85", lw=0.8)
axes[0].set_xlabel("Actual log10 FC"); axes[0].set_ylabel("Estimated log10 FC"); axes[0].set_title("Actual vs estimated (grey lines = class limits)")
axes[1].scatter(p_best, y_log - p_best, s=12, alpha=0.5); axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("Estimated log10 FC"); axes[1].set_ylabel("Error (actual − estimated)"); axes[1].set_title("Residuals")
plt.tight_layout(); plt.show()

ts = pd.DataFrame({"Month": test.Month.values, "actual": y_log, "estimate": p_best, "last month": test[LAST].values}).groupby("Month").mean()
ax = ts.plot(figsize=(11, 3.5), style=["k-o", "b-s", "r:"], ms=4)
ax.set_ylabel("log10 FC, mean of stations"); ax.set_title("Monthly lake-wide fecal coliform: actual vs estimated (test years)"); plt.show()

In [ ]:
# Accuracy per station (test years, log10 scale)
per_station = (pd.DataFrame({"Station": test.Station.values, "y": y_log, "p": p_best, "last": test[LAST].values})
               .groupby("Station").apply(lambda g: pd.Series({"n": len(g), "RMSE_log": np.sqrt(mean_squared_error(g.y, g.p)),
                                                              "RMSE_log repeat last month": np.sqrt(mean_squared_error(g.y, g.last)),
                                                              "median FC": 10 ** g.y.median()})))
per_station.sort_values("RMSE_log").round(3)

In [ ]:
# Which inputs matter most? Permutation importance on the test years.
target_te = test["dlogFC"] if best_change else test["logFC"]
imp = permutation_importance(fitted[BEST], test[best_cols], target_te, n_repeats=20, random_state=SEED,
                             scoring="neg_root_mean_squared_error")
importance = pd.Series(imp.importances_mean, index=best_cols)
grouped = pd.concat([importance[[c for c in best_cols if not c.startswith("stn_")]],
                     pd.Series({"station (all 15 together)": importance[[c for c in best_cols if c.startswith("stn_")]].sum()})])
ax = grouped.sort_values().plot.barh(figsize=(7, 4), color="#2e7d32")
ax.set_xlabel("Increase in RMSE (log10) when shuffled"); ax.set_title(f"Input importance — {best.model}, {best.version}"); plt.show()

## 7. DENR class check
Each estimate is converted back to MPN/100 mL, then to its DENR class, and compared with the class of the actual lab value.

In [ ]:
actual_cls = test.FC.map(fc_class)
est_cls = pd.Series(10 ** p_best, index=test.index).map(fc_class)
last_cls = pd.Series(10 ** test[LAST].values, index=test.index).map(fc_class)
cm = pd.DataFrame(confusion_matrix(actual_cls, est_cls, labels=CLASSES),
                  index=[f"actual {c}" for c in CLASSES], columns=[f"estimated {c}" for c in CLASSES])
print(cm, "\n")
order = {c: i for i, c in enumerate(CLASSES)}
within_one = (actual_cls.map(order) - est_cls.map(order)).abs() <= 1
print(f"Class accuracy: {100 * (actual_cls == est_cls).mean():.1f}%   (same class as last month: {100 * (actual_cls == last_cls).mean():.1f}%)")
print(f"Within one class: {100 * within_one.mean():.1f}%")

## 8. Year-by-year (rolling) test
Each year from 2018 to 2025 is predicted by models trained only on the years before it (versions C and D, settings found above). Note 2018: fecal coliform levels fell about 35-fold that year, so models trained on 2016–2017 alone struggle.

In [ ]:
roll = []
for year in range(2018, 2026):
    tr = df[df.Month.dt.year < year]
    te = df[(df.Month.dt.year == year) & (~df[FILLED])]
    row = {"year": year, "n": len(te), "Repeat last month": r2_score(te.logFC, te[LAST])}
    for vname in ["C: + last month's FC", "D: change from last month"]:
        cols, change = VERSIONS[vname]
        for mname in MODELS:
            _, p = fit_predict(fitted[(vname, mname)], cols, change, tr, te)
            row[f"{vname[0]} {mname}"] = r2_score(te.logFC, p)
    roll.append(row)
roll = pd.DataFrame(roll).set_index("year")
print("R² (log10) per year:")
print(roll.round(2))
print("\nMean R² over the years:")
print(roll.drop(columns="n").mean().round(3))

## 9. Comparison: random 70:30 split (manuscript method)
A random split puts neighbouring months of the same station in both training and test data, which makes scores look better than they will be on future months.

In [ ]:
rand_rows = []
tr, te = train_test_split(df, test_size=0.30, stratify=df.Station, random_state=SEED)   # stratified by station
te = te[~te[FILLED]]
for vname, (cols, change) in VERSIONS.items():
    for mname in MODELS:
        _, p = fit_predict(fitted[(vname, mname)], cols, change, tr, te)
        rand_rows.append({"version": vname, "model": mname, "R2_log random split": r2_score(te.logFC, p),
                          "R2_log time split": results.set_index(["version", "model"]).loc[(vname, mname), "R2_log"]})
pd.DataFrame(rand_rows).round(3)

## 10. Benchmarks (Chapter 1)
- **Benchmark 2:** at least one model reaches R² ≥ 0.55 on log10 fecal coliform on the held-out test set.
- **Benchmark 3:** the best ensemble beats MLR by ≥ 15% lower RMSE **and** ≥ 0.10 higher R², with a paired t-test on the cross-validation fold errors. The manuscript computes fecal coliform RMSE on MPN/100 mL; the log10 version is shown too because a few very high counts dominate the MPN RMSE.

In [ ]:
bench = []
for vname in VERSIONS:
    r = results[results.version == vname].set_index("model")
    ens = r.loc[["Random Forest", "XGBoost"]].sort_values("R2_log", ascending=False).index[0]
    cut_mpn = (r.loc["MLR", "RMSE_MPN"] - r.loc[ens, "RMSE_MPN"]) / r.loc["MLR", "RMSE_MPN"]
    cut_log = (r.loc["MLR", "RMSE_log"] - r.loc[ens, "RMSE_log"]) / r.loc["MLR", "RMSE_log"]
    gain = r.loc[ens, "R2_log"] - r.loc["MLR", "R2_log"]
    t = ttest_rel(folds[(vname, "MLR")], folds[(vname, ens)])
    bench.append({"version": vname, "best R2_log": r.R2_log.max(), "Benchmark 2 met (>= 0.55)": r.R2_log.max() >= 0.55,
                  "best ensemble": ens, "RMSE cut vs MLR % (MPN)": 100 * cut_mpn, "RMSE cut vs MLR % (log)": 100 * cut_log,
                  "R2 gain vs MLR": gain, "Benchmark 3 met": (cut_mpn >= 0.15) and (gain >= 0.10), "paired t-test p": t.pvalue})
pd.DataFrame(bench).round(3)

## 11. Final model for the dashboard
The best model is refit on **all** years (2016–2025) so it uses the most recent data, then saved. Its accuracy is the one measured on 2024–2025 above.

In [ ]:
final, _ = fit_predict(fitted[BEST], best_cols, best_change, df, df.iloc[:1])
path = os.path.join(OUT_DIR, "fc_model.joblib")
joblib.dump({"model": final, "features": best_cols, "predicts_change": best_change,
             "test_scores": best.to_dict(), "settings": params[BEST]}, path)
print("Saved:", path)

# Example: one station-month from the test years
example = test.iloc[[0]]
out = final.predict(example[best_cols])[0]
est_log = example[LAST].iloc[0] + out if best_change else out
print("\nExample input:")
print(example[["Month", "Station"] + READINGS].to_string(index=False))
print(f"Last month's fecal coliform: {10 ** example[LAST].iloc[0]:.0f} MPN/100 mL")
print(f"Estimated fecal coliform: {10 ** est_log:.0f} MPN/100 mL, Class {fc_class(10 ** est_log)}   "
      f"(actual lab value: {example.FC.iloc[0]:.0f}, Class {fc_class(example.FC.iloc[0])})")
print("Note: this example was part of the final model's training data; the honest accuracy is the 2024–2025 test above.")

In [ ]:
# Summary
base = results[results.version == "Baseline"].set_index("model")
print(f"Best fecal coliform model: {best.model} with inputs '{best.version}'")
print(f"Test 2024–2025: R² (log) {best.R2_log:.3f}, RMSE (log) {best.RMSE_log:.3f}, skill vs repeat last month {best.skill_vs_last_month:+.3f}")
print(f"Above-standard months caught {best['above_std_caught_%']:.0f}%, false alarms {best['false_alarms_%']:.1f}%")
print(f"Baselines: guess the average R² {base.loc['Guess the average', 'R2_log']:.3f}, repeat last month R² {base.loc['Repeat last month', 'R2_log']:.3f}")